# 思维树和LATS：刻意搜索

单挑思维链轨迹没有回溯的余地。思维树将推理问题转换成每个节点处自我评估的树结构。LATS（Language Agent Tree Search）将思维树和ReAct以及Reflexion通过蒙特卡洛树搜索。

## 问题描述

思维链是一次线性行进。如果第一步就错了，接下来的每一步都在基于错误的假设工作。在24点游戏上（给4个数字，通过加减乘除得到24），GPT-4 CoT只有4%的准确率。模型在早期选择了错误的子表达式，然后没办法还原了。

推理需要的是能够提出多个候选的能力、对它们进行评估，然后选择最有希望的那个，并且在碰到死路的时候能够进行回溯。这就是搜索。思维树和语言智能体树搜索是两种经典的格式。

## 基本概念

### 思维树

每个节点是一个连贯的中间步骤。每个节点又可以生出K个子节点，LLM基于评分提示词对每个节点做自我评估。摸索这个树————BFS、DFS或者束状搜索。

自我评估是承重部件。原始论文中展示了三种变体：
- `sure/likely/impossible` 三分类
- 1...10 数值分数
- 在候选者之间投票

这三种都在24游戏上击败了CoT。

### 语言模型树状搜索

LATS 将ToT、ReAct以及Reflexion 在MCTS（Monte Carlo Tree Search）上结合起来。LLM扮演三种角色：
- 策略。推举候选者（ReAct-Style）
- 价值评估。 对一部分轨迹打分（ToT-Style 自我评估）
- 自我反射。 在失败的时候，通过自然语言写反思（Reflexion-Style）然后作用于后续推理。

环境反馈（观察）被混进价值函数，所以搜索被真实的工具结果启发，而不是仅基于模型的意见。


# 开始编码

本章核心：**ToT（多候选 + 自我评估 + 树搜索）** 与 **LATS（MCTS = 策略扩展 + 价值评估 + 可选反思）**。  
用经典 **24 点** 当环境：先玩具 BFS/束搜索与迷你 MCTS；再用 **PyTorch 价值网**；最后用 **LangGraph + DeepSeek** 做生产示意。


## 1. 教学玩具：24 点环境 + ToT 束搜索


In [ ]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable

from typing_extensions import Literal


class Viability(str, Enum):
    """ToT 三分类自我评估。"""

    SURE = "sure"
    LIKELY = "likely"
    IMPOSSIBLE = "impossible"


VIABILITY_SCORE: dict[Viability, float] = {
    Viability.SURE: 1.0,
    Viability.LIKELY: 0.5,
    Viability.IMPOSSIBLE: 0.0,
}


@dataclass(frozen=True)
class Game24State:
    """24 点状态：剩余数字（已排序，便于去重）。"""

    nums: tuple[float, ...]

    @staticmethod
    def from_list(nums: list[float] | list[int]) -> Game24State:
        """
        Args:
            nums: 剩余数字。

        Returns:
            state: 规范化状态。
        """
        return Game24State(tuple(sorted(float(x) for x in nums)))

    def is_success(self, target: float = 24.0, eps: float = 1e-6) -> bool:
        """
        Args:
            target: 目标值。
            eps: 浮点容差。

        Returns:
            ok: 是否只剩一个数且等于目标。
        """
        return len(self.nums) == 1 and abs(self.nums[0] - target) < eps


@dataclass(frozen=True)
class Move:
    """合并两个数的一步操作。"""

    i: int
    j: int
    op: Literal["+", "-", "*", "/"]
    left: float
    right: float
    result: float

    def describe(self) -> str:
        """
        Returns:
            text: 人类可读步骤，如 ``6*4=24``。
        """
        return f"{self.left:g}{self.op}{self.right:g}={self.result:g}"


def apply_op(a: float, b: float, op: str) -> float | None:
    """
    Args:
        a: 左操作数。
        b: 右操作数。
        op: ``+ - * /``。

    Returns:
        result: 结果；非法除零返回 ``None``。
    """
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op == "*":
        return a * b
    if op == "/":
        if abs(b) < 1e-12:
            return None
        return a / b
    raise ValueError(f"unknown op {op}")


def legal_moves(state: Game24State) -> list[tuple[Move, Game24State]]:
    """
    枚举所有合法下一步（无序对 + 四种运算；减法/除法考虑顺序）。

    Args:
        state: 当前状态。

    Returns:
        moves: ``(Move, next_state)`` 列表。
    """
    nums = list(state.nums)
    n = len(nums)
    out: list[tuple[Move, Game24State]] = []
    seen: set[tuple[float, ...]] = set()
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            for op in ("+", "-", "*", "/"):
                # ``+`` ``*`` 交换律：只保留 i<j 避免重复
                if op in ("+", "*") and i > j:
                    continue
                res = apply_op(nums[i], nums[j], op)
                if res is None:
                    continue
                rest = [nums[k] for k in range(n) if k != i and k != j]
                rest.append(res)
                nxt = Game24State.from_list(rest)
                if nxt.nums in seen:
                    continue
                seen.add(nxt.nums)
                move = Move(i, j, op, nums[i], nums[j], res)  # type: ignore[arg-type]
                out.append((move, nxt))
    return out


def heuristic_viability(state: Game24State, target: float = 24.0) -> Viability:
    """
    玩具自我评估（无 LLM）：根据与 24 的距离/规模粗分三档。

    Args:
        state: 节点状态。
        target: 目标。

    Returns:
        label: sure / likely / impossible。
    """
    if state.is_success(target):
        return Viability.SURE
    if not state.nums:
        return Viability.IMPOSSIBLE
    if len(state.nums) == 1:
        return Viability.LIKELY if abs(state.nums[0] - target) < 8 else Viability.IMPOSSIBLE
    mx = max(abs(x) for x in state.nums)
    mn = min(abs(x) for x in state.nums)
    if mx > target * 30 and mn >= 1:
        return Viability.IMPOSSIBLE
    return Viability.LIKELY


def heuristic_numeric_score(state: Game24State, target: float = 24.0) -> float:
    """
    1...10 数值分。注意：中间结果里「碰巧出现 24」但还有其它数时，不能给高分。

    Args:
        state: 状态。
        target: 目标。

    Returns:
        score: ``[0, 10]``。
    """
    if state.is_success(target):
        return 10.0
    if heuristic_viability(state, target) is Viability.IMPOSSIBLE:
        return 0.5
    if len(state.nums) == 1:
        return float(max(1.0, 10.0 - abs(state.nums[0] - target)))
    # 更少剩余数字更好
    length_bonus = (4 - len(state.nums)) * 1.5
    # 若已有一个 24 但仍有其它数，只给中等分（还需消掉其它数）
    if any(abs(x - target) < 1e-6 for x in state.nums):
        return float(3.0 + length_bonus)
    d = min(abs(x - target) for x in state.nums)
    return float(max(1.0, 4.5 + length_bonus - 0.12 * d))


def solve_game24_bfs(start: Game24State, target: float = 24.0) -> ThoughtNode | None:
    """
    穷尽 BFS（对照基线）：保证小牌面可解时能找到路径。

    Args:
        start: 初始状态。
        target: 目标。

    Returns:
        node: 成功叶；无解 ``None``。
    """
    from collections import deque

    root = ThoughtNode(state=start)
    q: deque[ThoughtNode] = deque([root])
    seen: set[tuple[float, ...]] = {start.nums}
    while q:
        node = q.popleft()
        if node.state.is_success(target):
            return node
        for move, nxt_state in legal_moves(node.state):
            if nxt_state.nums in seen:
                continue
            seen.add(nxt_state.nums)
            child = ThoughtNode(state=nxt_state, move=move, parent=node)
            node.children.append(child)
            q.append(child)
    return None


@dataclass
class ThoughtNode:
    """ToT / 搜索树节点。"""

    state: Game24State
    move: Move | None = None
    parent: ThoughtNode | None = None
    children: list[ThoughtNode] = field(default_factory=list)
    viability: Viability = Viability.LIKELY
    value: float = 0.0
    """ToT 评估分或 MCTS 平均价值。"""

    # MCTS 统计
    n_visits: int = 0
    w_sum: float = 0.0

    def path_moves(self) -> list[Move]:
        """
        Returns:
            moves: 从根到当前的操作序列。
        """
        nodes: list[ThoughtNode] = []
        cur: ThoughtNode | None = self
        while cur is not None:
            nodes.append(cur)
            cur = cur.parent
        nodes.reverse()
        return [n.move for n in nodes if n.move is not None]


def tot_beam_search(
    start: Game24State,
    *,
    beam_width: int = 8,
    max_depth: int = 6,
    target: float = 24.0,
    scorer: Callable[[Game24State], float] | None = None,
) -> ThoughtNode | None:
    """
    ToT 风格束搜索：每层展开合法子节点，按自我评估分保留 top-``beam_width``。

    Args:
        start: 初始牌面。
        beam_width: 束宽 ``K``。
        max_depth: 最大深度。
        target: 目标值。
        scorer: 节点打分；默认 ``heuristic_numeric_score``。

    Returns:
        best: 成功叶节点；找不到则 ``None``。
    """
    score_fn = scorer or (lambda s: heuristic_numeric_score(s, target))
    root = ThoughtNode(state=start, value=score_fn(start))
    beam: list[ThoughtNode] = [root]
    seen: set[tuple[float, ...]] = {start.nums}

    for _ in range(max_depth):
        nxt: list[ThoughtNode] = []
        for node in beam:
            if node.state.is_success(target):
                return node
            for move, state in legal_moves(node.state):
                if state.nums in seen and not state.is_success(target):
                    continue
                child = ThoughtNode(
                    state=state,
                    move=move,
                    parent=node,
                    viability=heuristic_viability(state, target),
                    value=score_fn(state),
                )
                if child.viability is Viability.IMPOSSIBLE:
                    continue
                if state.is_success(target):
                    node.children.append(child)
                    return child
                seen.add(state.nums)
                node.children.append(child)
                nxt.append(child)
        if not nxt:
            break
        nxt.sort(key=lambda n: n.value, reverse=True)
        beam = nxt[:beam_width]
    return None



def tot_dfs(
    start: Game24State,
    *,
    max_depth: int = 6,
    target: float = 24.0,
    scorer: Callable[[Game24State], float] | None = None,
) -> ThoughtNode | None:
    """
    ToT 风格 DFS：子节点按自评排序后依次尝试，失败则回溯。

    Args:
        start: 初始状态。
        max_depth: 最大深度。
        target: 目标。
        scorer: 排序用分数。

    Returns:
        node: 成功叶；失败 ``None``。
    """
    score_fn = scorer or (lambda s: heuristic_numeric_score(s, target))
    root = ThoughtNode(state=start, value=score_fn(start))

    def rec(node: ThoughtNode, depth: int) -> ThoughtNode | None:
        if node.state.is_success(target):
            return node
        if depth >= max_depth:
            return None
        cands: list[ThoughtNode] = []
        for move, state in legal_moves(node.state):
            if heuristic_viability(state, target) is Viability.IMPOSSIBLE:
                continue
            child = ThoughtNode(
                state=state,
                move=move,
                parent=node,
                viability=heuristic_viability(state, target),
                value=score_fn(state),
            )
            cands.append(child)
        cands.sort(key=lambda n: n.value, reverse=True)
        for child in cands:
            node.children.append(child)
            found = rec(child, depth + 1)
            if found is not None:
                return found
        return None

    return rec(root, 0)



def format_solution(node: ThoughtNode) -> str:
    """
    Args:
        node: 成功节点。

    Returns:
        text: 步骤串。
    """
    moves = node.path_moves()
    return " → ".join(m.describe() for m in moves)


print("Game24 + ToT beam ready")


## 2. 玩具 LATS：迷你 MCTS（UCB + 扩展 + 回传）


In [ ]:
def ucb1(parent: ThoughtNode, child: ThoughtNode, c: float = 1.4) -> float:
    """
    Args:
        parent: 父节点。
        child: 子节点。
        c: 探索系数。

    Returns:
        score: UCB1 分数；未访问子节点为 ``+inf``。
    """
    if child.n_visits == 0:
        return float("inf")
    exploit = child.w_sum / child.n_visits
    explore = c * math.sqrt(math.log(parent.n_visits + 1) / child.n_visits)
    return exploit + explore


def mcts_select(root: ThoughtNode) -> ThoughtNode:
    """
    沿 UCB 走到第一个未完全展开或叶节点。

    Args:
        root: 根。

    Returns:
        node: 选中节点。
    """
    node = root
    while node.children:
        # 若还有未生成的合法走法，停在此节点扩展
        legal = legal_moves(node.state)
        if len(node.children) < len(legal):
            return node
        node = max(node.children, key=lambda ch: ucb1(node, ch))
    return node


def mcts_expand(node: ThoughtNode, target: float = 24.0) -> ThoughtNode:
    """
    策略角色：从未试过的合法走法中扩展一个子节点。

    Args:
        node: 待扩展节点。
        target: 目标。

    Returns:
        child: 新子节点；若无可扩展则返回自身。
    """
    existing = {c.move.describe() for c in node.children if c.move}
    for move, state in legal_moves(node.state):
        if move.describe() in existing:
            continue
        child = ThoughtNode(
            state=state,
            move=move,
            parent=node,
            viability=heuristic_viability(state, target),
            value=0.0,
        )
        node.children.append(child)
        return child
    return node


def mcts_value(state: Game24State, target: float = 24.0) -> float:
    """
    价值角色（ToT 风格玩具）：把启发式分映射到 ``[0,1]``。

    Args:
        state: 状态。
        target: 目标。

    Returns:
        v: 价值估计。
    """
    if state.is_success(target):
        return 1.0
    return heuristic_numeric_score(state, target) / 10.0


def mcts_backup(node: ThoughtNode, value: float) -> None:
    """
    沿父链回传。

    Args:
        node: 起点。
        value: 叶价值。
    """
    cur: ThoughtNode | None = node
    while cur is not None:
        cur.n_visits += 1
        cur.w_sum += value
        cur.value = cur.w_sum / max(cur.n_visits, 1)
        cur = cur.parent


def mcts_rollout(
    state: Game24State,
    *,
    target: float = 24.0,
    max_depth: int = 6,
    rng: random.Random | None = None,
) -> float:
    """
    ε-greedy rollout（多数时候跟启发式走），返回 ``0/1`` 价值。

    Args:
        state: 起始状态。
        target: 目标。
        max_depth: 最大步数。
        rng: 随机源。

    Returns:
        value: 成功 1.0 否则 0.0。
    """
    rng = rng or random.Random(0)
    cur = state
    for _ in range(max_depth):
        if cur.is_success(target):
            return 1.0
        moves = legal_moves(cur)
        if not moves:
            return 0.0
        if rng.random() < 0.25:
            _, cur = rng.choice(moves)
        else:
            scored = [
                (mcts_value(nxt, target), nxt)
                for _, nxt in moves
                if heuristic_viability(nxt, target) is not Viability.IMPOSSIBLE
            ]
            if not scored:
                return 0.0
            scored.sort(key=lambda t: t[0], reverse=True)
            cur = scored[0][1]
    return 1.0 if cur.is_success(target) else 0.0


def reflect_on_failure(path: list[Move], final: Game24State) -> str:
    """
    Reflexion 风格玩具反思（失败路径）。

    Args:
        path: 走法。
        final: 终态。

    Returns:
        text: 自然语言反思。
    """
    steps = " → ".join(m.describe() for m in path) or "<empty>"
    return (
        f"Path [{steps}] ended at {final.nums}. "
        "Prefer keeping factors of 24 (e.g. 8,6,4,3); avoid huge intermediates."
    )

def lats_mcts(
    start: Game24State,
    *,
    n_simulations: int = 64,
    target: float = 24.0,
    seed: int = 0,
) -> tuple[ThoughtNode | None, list[str]]:
    """
    迷你 LATS：MCTS（UCB 选择 → 扩展 → 启发式 rollout 并物化路径 → 回传）。

    Args:
        start: 初始状态。
        n_simulations: 模拟次数。
        target: 目标。
        seed: 随机种子。

    Returns:
        best: 成功节点或访问最多节点。
        reflections: 失败反思。
    """
    rng = random.Random(seed)
    root = ThoughtNode(state=start)
    reflections: list[str] = []

    def walk(n: ThoughtNode) -> list[ThoughtNode]:
        nodes = [n]
        for c in n.children:
            nodes.extend(walk(c))
        return nodes

    for _ in range(n_simulations):
        leaf = mcts_select(root)
        if leaf.state.is_success(target):
            mcts_backup(leaf, 1.0)
            return leaf, reflections

        # 扩展一个新子节点（策略）
        existing = {c.move.describe() for c in leaf.children if c.move}
        moves = legal_moves(leaf.state)
        rng.shuffle(moves)
        child: ThoughtNode | None = None
        for move, state in moves:
            if move.describe() in existing:
                continue
            child = ThoughtNode(
                state=state,
                move=move,
                parent=leaf,
                viability=heuristic_viability(state, target),
            )
            leaf.children.append(child)
            break
        if child is None:
            mcts_backup(leaf, mcts_value(leaf.state, target))
            continue

        # 从新节点启发式向下物化一条路径（把 rollout 写进树，便于取出解）
        node = child
        for _ply in range(5):
            if node.state.is_success(target):
                break
            if heuristic_viability(node.state, target) is Viability.IMPOSSIBLE:
                reflections.append(reflect_on_failure(node.path_moves(), node.state))
                break
            opts = []
            for move, state in legal_moves(node.state):
                if heuristic_viability(state, target) is Viability.IMPOSSIBLE:
                    continue
                opts.append((mcts_value(state, target), move, state))
            if not opts:
                break
            opts.sort(key=lambda t: t[0], reverse=True)
            # ε-greedy
            pick = opts[0] if rng.random() > 0.2 else opts[rng.randrange(len(opts))]
            _, move, state = pick
            # 若已有相同子，复用
            found_existing = None
            for ch in node.children:
                if ch.move and ch.move.describe() == move.describe():
                    found_existing = ch
                    break
            if found_existing is None:
                found_existing = ThoughtNode(
                    state=state,
                    move=move,
                    parent=node,
                    viability=heuristic_viability(state, target),
                )
                node.children.append(found_existing)
            node = found_existing

        if node.state.is_success(target):
            mcts_backup(node, 1.0)
            return node, reflections
        mcts_backup(node, mcts_value(node.state, target))

    for n in walk(root):
        if n.state.is_success(target):
            return n, reflections
    all_nodes = walk(root)
    if not all_nodes:
        return None, reflections
    best = max(all_nodes, key=lambda n: (n.n_visits, n.value))
    return best, reflections


print("LATS mini-MCTS ready")


## 3. 玩具示例：ToT 束搜索 vs LATS


In [ ]:
def demo_tot_and_lats() -> None:
    """对照 BFS / ToT-DFS / LATS-MCTS。"""
    puzzles = [
        [8, 8, 3, 3],
        [1, 3, 4, 6],
        [4, 9, 10, 13],
    ]
    for cards in puzzles:
        start = Game24State.from_list(cards)
        print(f"\n=== cards {cards} ===")
        bfs = solve_game24_bfs(start)
        print("BFS:", format_solution(bfs) if bfs and bfs.state.is_success() else "failed")

        dfs = tot_dfs(start, max_depth=6)
        print("ToT-DFS:", format_solution(dfs) if dfs and dfs.state.is_success() else "failed")

        beam = tot_beam_search(start, beam_width=16, max_depth=6)
        print("ToT-Beam:", format_solution(beam) if beam and beam.state.is_success() else "failed")

        lat, refs = lats_mcts(start, n_simulations=400, seed=2)
        if lat and lat.state.is_success():
            print("LATS:", format_solution(lat), f"(visits={lat.n_visits})")
        else:
            print("LATS: incomplete", lat.state.nums if lat else None)
        if refs:
            print("reflections[0]:", refs[0][:100], "...")

    s = Game24State.from_list([8, 8, 3, 3])
    assert solve_game24_bfs(s) is not None
    assert tot_dfs(s) is not None and tot_dfs(s).state.is_success()
    print("\nTOY DEMOS OK")


demo_tot_and_lats()


## 4. PyTorch：状态价值网（替代/增强自我评估）

把剩余数字 pad 成固定维特征，回归 ``[0,1]`` 价值，供 ToT/MCTS 打分。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def encode_state(state: Game24State, dim: int = 4) -> torch.Tensor:
    """
    Args:
        state: 24 点状态。
        dim: 固定长度（右侧 0 填充）。

    Returns:
        x: ``(dim,)``。
    """
    xs = list(state.nums)[:dim]
    while len(xs) < dim:
        xs.append(0.0)
    # 粗归一化
    return torch.tensor([x / 24.0 for x in xs], dtype=torch.float32)


class Game24ValueNet(nn.Module):
    """状态 → 价值 logit（sigmoid 后为成功潜力）。"""

    def __init__(self, dim: int = 4) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, dim)`` 或 ``(dim,)``。

        Returns:
            logit: ``(B,)`` 或标量。
        """
        single = x.ndim == 1
        if single:
            x = x.unsqueeze(0)
        y = self.net(x).squeeze(-1)
        return y.squeeze(0) if single else y

    @torch.no_grad()
    def value(self, state: Game24State) -> float:
        """
        Args:
            state: 状态。

        Returns:
            v: ``[0,1]``。
        """
        logit = self.forward(encode_state(state))
        return float(torch.sigmoid(logit).item())


def train_value_net(steps: int = 400, lr: float = 0.02) -> Game24ValueNet:
    """
    用启发式标签做蒸馏式回归/分类。

    Args:
        steps: 步数。
        lr: 学习率。

    Returns:
        model: 训练后的价值网。
    """
    # 合成状态
    samples: list[tuple[Game24State, float]] = []
    samples.append((Game24State.from_list([24]), 1.0))
    samples.append((Game24State.from_list([24.0]), 1.0))
    samples.append((Game24State.from_list([8, 3]), 0.7))
    samples.append((Game24State.from_list([6, 4]), 0.75))
    samples.append((Game24State.from_list([1, 1, 1, 1]), 0.2))
    samples.append((Game24State.from_list([100, 100]), 0.05))
    samples.append((Game24State.from_list([8, 8, 3, 3]), 0.6))
    samples.append((Game24State.from_list([5]), 0.15))

    model = Game24ValueNet()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for st, y in samples:
            logit = model(encode_state(st))
            target = torch.tensor(y)
            loss = loss + F.binary_cross_entropy_with_logits(logit, target)
        loss = loss / len(samples)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


def demo_value_net() -> None:
    """价值网应给成功态更高分。"""
    torch.manual_seed(0)
    net = train_value_net()
    v_ok = net.value(Game24State.from_list([24]))
    v_bad = net.value(Game24State.from_list([100, 100]))
    print("=== value net ===")
    print(f"V([24])={v_ok:.3f} V([100,100])={v_bad:.3f}")
    assert v_ok > v_bad

    # 用价值网作为 ToT scorer
    start = Game24State.from_list([8, 8, 3, 3])
    node = tot_dfs(
        start,
        max_depth=6,
        scorer=lambda s: net.value(s) * 10.0,
    )
    print("ToT-DFS+NN:", format_solution(node) if node and node.state.is_success() else "fail")
    assert node is not None and node.state.is_success()
    print("VALUE NET OK")


demo_value_net()


## 5. 生产级：LangGraph ToT 一步扩展 + DeepSeek 评估

示意生产形态：对当前状态让 LLM **提出 K 个思维/操作**，再 **sure/likely/impossible** 打分，保留最优继续（束宽=1 的深度推进）。完整 LATS 成本高，这里聚焦可跑通的 ToT 核心。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


class ThoughtCandidate(BaseModel):
    """策略提出的一个中间思维（操作描述）。"""

    expression: str = Field(description="e.g. '8*3=24' using two current numbers")
    remaining: list[float] = Field(description="numbers left after this step")
    note: str = Field(default="", description="brief rationale")


class ProposeOut(BaseModel):
    """一次扩展的 K 个候选。"""

    candidates: list[ThoughtCandidate]


class JudgeOut(BaseModel):
    """自我评估。"""

    label: Literal["sure", "likely", "impossible"]
    score: float = Field(ge=0.0, le=10.0)
    reason: str = ""


class TotState(TypedDict, total=False):
    """LangGraph ToT 状态。"""

    cards: list
    target: float
    depth: int
    max_depth: int
    beam_k: int
    path: list
    current: list
    done: bool
    success: bool
    trace: list
    _candidates: list


def get_llm(*, temperature: float = 0.2) -> Any:
    """
    Returns:
        llm: DeepSeek 模型。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def propose_node(state: TotState) -> dict[str, Any]:
    """
    策略：为当前剩余数字提出 ``beam_k`` 个候选步骤。

    Args:
        state: ToT 状态。

    Returns:
        update: 含 ``_candidates``。
    """
    k = int(state.get("beam_k", 3))
    current = list(state.get("current") or state["cards"])
    llm = get_llm().with_structured_output(ProposeOut)
    out: ProposeOut = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are the Tree-of-Thoughts policy for Game 24. "
                    "Given remaining numbers, propose K distinct next steps. "
                    "Each step combines exactly two numbers with +,-,*,/ "
                    "and returns the new remaining multiset. Avoid clearly dead ends (e.g. reducing to numbers that cannot make 24)."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "remaining": current,
                        "target": state.get("target", 24),
                        "K": k,
                        "path_so_far": state.get("path", []),
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    return {"_candidates": [c.model_dump() for c in out.candidates[:k]]}


def evaluate_and_select_node(state: TotState) -> dict[str, Any]:
    """
    价值：对每个候选打分，选最优推进；成功则结束。

    Args:
        state: 含 ``_candidates``。

    Returns:
        update: path/current/done/success 等。
    """
    cands: list = list(state.get("_candidates") or [])
    if not cands:
        return {
            "done": True,
            "success": False,
            "trace": list(state.get("trace") or []) + ["no candidates"],
        }

    llm = get_llm(temperature=0.0).with_structured_output(JudgeOut)
    scored: list[tuple[float, dict[str, Any], JudgeOut]] = []
    for c in cands:
        remaining = [float(x) for x in c.get("remaining") or []]
        # 环境反馈：本地 BFS 可解性（LATS 要点：不全信模型自评）
        env_bonus = 0.0
        try:
            if solve_game24_bfs(Game24State.from_list(remaining)) is not None:
                env_bonus = 5.0
            elif len(remaining) == 1 and abs(remaining[0] - float(state.get("target", 24))) < 1e-4:
                env_bonus = 10.0
        except Exception:
            env_bonus = 0.0

        judge: JudgeOut = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are the ToT value model. Classify whether this partial "
                        "Game-24 state can reach the target: sure/likely/impossible. "
                        "Also give score 1..10. Prefer steps that keep flexibility."
                    )
                ),
                HumanMessage(
                    content=json.dumps(
                        {
                            "target": state.get("target", 24),
                            "candidate": c,
                            "path_so_far": state.get("path", []),
                            "env_solvable_hint": env_bonus >= 5.0,
                        },
                        ensure_ascii=False,
                    )
                ),
            ]
        )
        if judge.label == "impossible" and env_bonus < 5.0:
            continue
        scored.append((float(judge.score) + env_bonus, c, judge))

    trace = list(state.get("trace") or [])
    if not scored:
        return {
            "done": True,
            "success": False,
            "trace": trace + ["all candidates judged impossible"],
        }

    scored.sort(key=lambda t: t[0], reverse=True)
    best_score, best, judge = scored[0]
    path = list(state.get("path") or []) + [str(best.get("expression", ""))]
    remaining = [float(x) for x in best.get("remaining") or []]
    trace.append(
        f"pick {best.get('expression')} -> {remaining} "
        f"[{judge.label} score={best_score:.1f}] {judge.reason}"
    )

    success = (
        len(remaining) == 1
        and abs(remaining[0] - float(state.get("target", 24))) < 1e-4
    )
    depth = int(state.get("depth", 0)) + 1
    done = bool(success or depth >= int(state.get("max_depth", 4)))
    return {
        "path": path,
        "current": remaining,
        "depth": depth,
        "done": done,
        "success": success,
        "trace": trace,
        "_candidates": [],
    }


def route_tot(state: TotState) -> str:
    """
    Args:
        state: 状态。

    Returns:
        next: ``propose`` 或 ``end``。
    """
    if state.get("done"):
        return "end"
    return "propose"


def build_tot_graph() -> Any:
    """
    Returns:
        app: 编译图。
    """
    g: StateGraph = StateGraph(TotState)
    g.add_node("propose", propose_node)
    g.add_node("select", evaluate_and_select_node)
    g.add_edge(START, "propose")
    g.add_edge("propose", "select")
    g.add_conditional_edges("select", route_tot, {"propose": "propose", "end": END})
    return g.compile()


def run_tot_game24(
    cards: list[float],
    *,
    target: float = 24.0,
    beam_k: int = 3,
    max_depth: int = 4,
) -> TotState:
    """
    Args:
        cards: 初始牌。
        target: 目标。
        beam_k: 每层候选数。
        max_depth: 最大深度。

    Returns:
        state: 最终状态。
    """
    app = build_tot_graph()
    return app.invoke(
        {
            "cards": cards,
            "target": target,
            "depth": 0,
            "max_depth": max_depth,
            "beam_k": beam_k,
            "path": [],
            "current": list(cards),
            "done": False,
            "success": False,
            "trace": [],
        }
    )


print("LangGraph ToT ready |", MODEL)


## 6. 生产示例


In [ ]:
def demo_deepseek_tot() -> None:
    """需要网络与 API Key。优先选较易的 ``[8,8,3,3]``。"""
    out = run_tot_game24([8, 8, 3, 3], beam_k=3, max_depth=4)
    print("=== path ===", out.get("path"))
    print("=== current ===", out.get("current"))
    print("=== success ===", out.get("success"))
    print("=== trace ===")
    for line in out.get("trace") or []:
        print("-", line)
    # 不强制 assert success：LLM 偶发走错；有路径与 trace 即示意跑通
    assert out.get("path") is not None
    assert out.get("trace")


demo_deepseek_tot()
